# 02. Очистка данных

Каждый шаг записывается в журнал `log`: что сделали, почему и сколько строк затронуто.
Принцип: **строки-вакансии не удаляем без причины.** Если зарплата неправдоподобна,
обнуляем только зарплату, а вакансию оставляем: она нужна для анализа рынка
и для оценки «скрытых» зарплат.

Результат: `data/processed/vacancies_clean.csv` — вход для ML-репозитория.

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 100)

df = pd.read_excel("../data/raw/vacancy.xlsx")
df["published"] = pd.to_datetime(df["publication_time"], unit="s")
df["text"] = (df["name"].fillna("") + " " + df["requirements"].fillna("") + " "
              + df["responsibilities"].fillna("")).str.lower()

log = []
def note(step, why, rows):
    log.append({"шаг": step, "почему": why, "строк": int(rows)})

n0 = len(df)
print(n0)

13349


## 1. Объявления, которые не являются вакансиями

На lalafo в раздел «работа» попадают резюме («ищу работу») и объявления об услугах/аренде.
Это не спрос работодателя, их удаляем. Слово «резюме» специально **не** используем:
работодатели пишут «присылайте резюме».

In [2]:
seeker = df["text"].str.contains(r"\bищу работу|\bищу подработку|иш издейм|иш издеп жатам")
service = df["text"].str.contains(r"возьму в аренду|\bсдаю\b|\bсдается\b|\bпродаю\b|\bпрода[её]тся\b|отремонтируем")
print("Резюме:", seeker.sum(), "| Услуги/аренда:", service.sum())
df.loc[seeker | service, ["name", "source"]].sample(10, random_state=0)

Резюме: 184 | Услуги/аренда: 154


,name,source
6479,Парикмахер Универсал. Аренда места. БГУ,lalafo
11490,Лешмейкер. Аренда места. Моссовет,lalafo
12680,Лешмейкер. Аренда места,lalafo
8773,Мастер маникюра. Аренда места. Ошский рынок / базар,lalafo
9770,Ищу подработку на кухне на ночь после 23:00 до 05:00,lalafo
11565,Лешмейкер. Аренда места,lalafo
4506,Лешмейкер. Аренда места. 5 мкр,lalafo
11110,Няня,lalafo
6212,Сдаю кушетку для мастера перманентного макияжа и ресниц. либо на %,lalafo
11658,Ищу работу. Учитель начальных классов. Учусь на последнем курсе в,lalafo


In [3]:
note("Удалены резюме и объявления об услугах", "не вакансии, а предложения работников или услуги",
     (seeker | service).sum())
df = df[~(seeker | service)].copy()
len(df)

13011

## 2. Дубликаты

`id` уникальны, но одна и та же вакансия публикуется повторно (на lalafo — переподача объявления,
на hh — одна вакансия на несколько адресов). Дубликатом считаем совпадение
названия, текста требований и зарплаты.

In [4]:
dup_cols = ["name", "requirements", "compensation_from", "compensation_to"]
dup = df.duplicated(dup_cols, keep="last")      # оставляем самую свежую публикацию
print("Дубликатов:", dup.sum())
print(df[dup]["source"].value_counts())
note("Удалены дубликаты", "одинаковые название + требования + зарплата; оставлена последняя публикация",
     dup.sum())
df = df.sort_values("published")
df = df[~df.duplicated(dup_cols, keep="last")].copy()
len(df)

Дубликатов: 365
source
headhunter    184
lalafo        181
Name: count, dtype: int64


12646

## 3. Зарплата: одна целевая переменная

Диапазон бывает трёх видов: «от–до», только «от», только «до».
Правило: «от–до» → середина диапазона; только одна граница → она сама.
Тип записываем в `salary_type`: это пригодится модели (вакансии «от X» в среднем платят больше X).

In [5]:
f, t = df["compensation_from"], df["compensation_to"]
df["salary_raw"] = np.where(f.notna() & t.notna(), (f + t) / 2, f.fillna(t))
df["salary_type"] = np.select(
    [f.notna() & t.notna(), f.notna(), t.notna()], ["range", "from", "to"], default="hidden")
df["salary_type"].value_counts()

salary_type
hidden    6516
from      4800
range     1251
to          79
Name: count, dtype: int64

## 4. Валюта → сомы

Официальные курсы НБКР на 14.03.2024 (середина периода данных),
источник: economist.kg. 5 строк с зарплатой без валюты — объявления lalafo из Кыргызстана,
считаем их в сомах.

In [6]:
RATES = {"KGS": 1.0, "USD": 89.4870, "EUR": 97.7824, "RUR": 0.9785, "KZT": 0.1988}
has_raw = df["salary_raw"].notna()
no_cur = has_raw & df["compensation_currency"].isna()
note("Валюта не указана → KGS", "объявления lalafo из КР", no_cur.sum())
df.loc[no_cur, "compensation_currency"] = "KGS"

df["salary_kgs"] = df["salary_raw"] * df["compensation_currency"].map(RATES)
not_kgs = has_raw & (df["compensation_currency"] != "KGS")
note("Пересчёт USD/RUR/EUR/KZT в сомы", "курс НБКР на 14.03.2024", not_kgs.sum())
df.loc[has_raw, "compensation_currency"].value_counts()

compensation_currency
KGS    5752
USD     216
RUR     148
KZT       8
EUR       6
Name: count, dtype: int64

## 5. Оплата за день / неделю / час → в месяц

На lalafo много зарплат вида «1 500 сом за выход». Ключевые слова пишут по-разному
(«в день», «за смену», «выход», «в ден»), а «оплата еженедельно» часто означает
**частоту выплаты**, а не ставку. Поэтому основное правило — по самой сумме:

| Сумма | Решение | Почему |
|---|---|---|
| < 500 и «в час» | × 208 (8 ч × 26 дней) | почасовая ставка |
| 500 – 5 000 | × 26 | месячная зарплата в таком размере неправдоподобна, это дневная ставка |
| 5 000 – 30 000 и «в неделю» | × 4,33 | недельная ставка |

Допущение: 26 рабочих дней (график 6/1 типичен для таких вакансий). Остальное ниже 10 000
после пересчёта считаем выбросом (шаг 6).

In [7]:
hour_kw = r"в час\b|за час|почасов"
week_kw = r"в неделю|еженедельн|жумасына|за неделю"
s = df["salary_kgs"]
rules = {
    "hour": (df["text"].str.contains(hour_kw) & (s < 500), 8 * 26),
    "day":  (s.between(500, 5_000, inclusive="left"), 26),
    "week": (df["text"].str.contains(week_kw) & s.between(5_000, 30_000, inclusive="left"), 4.33),
}
df["pay_period"] = np.where(df["salary_kgs"].notna(), "month", None)
for p, (m, k) in rules.items():
    m = m & (df["pay_period"] == "month")
    df.loc[m, "pay_period"] = p
    df.loc[m, "salary_kgs"] = df.loc[m, "salary_kgs"] * k
    note(f"Ставка «{p}» пересчитана в месяц", f"× {k}", m.sum())

df["pay_period"].value_counts()

pay_period
month    5032
day      1029
week       65
hour        4
Name: count, dtype: int64

## 6. Выбросы зарплаты

Смотрим на распределение после всех пересчётов. Ниже 10 000 сом в месяц —
почти всегда неучтённая дневная оплата или опечатка («30» вместо «30 000»);
выше 600 000 — опечатки (встречается 1 000 000 000). Такие зарплаты **обнуляем**,
вакансию оставляем и помечаем `salary_outlier`.

In [8]:
s = df["salary_kgs"]
print(s.describe(percentiles=[.01, .02, .05, .5, .95, .98, .99]).round(0))
LOW, HIGH = 10_000, 600_000
out = s.notna() & ((s < LOW) | (s > HIGH))
print("Выбросов:", out.sum(), "| ниже:", (s < LOW).sum(), "| выше:", (s > HIGH).sum())
df.loc[out & (s > HIGH), ["name", "salary_raw", "compensation_currency", "source"]].head(10)

count    6.130000e+03
mean     2.457560e+05
std      1.288960e+07
min      1.000000e+00
1%       4.500000e+01
2%       1.500000e+02
5%       1.000000e+04
50%      3.750000e+04
95%      1.082500e+05
98%      1.500000e+05
99%      2.000000e+05
max      1.000000e+09
Name: salary_kgs, dtype: float64
Выбросов: 290 | ниже: 276 | выше: 14


,name,salary_raw,compensation_currency,source
13073,"FR510E2-HD Cummins двигатель, Kawasaki насос, клапан, г идромотор",2.800000e+05,USD,lalafo
7810,Практика автоэлектрика + теория. Обучение + практика. Если вы хотите,1.111111e+07,KGS,lalafo
9573,Парикмахер Колорист. Процент,1.000000e+09,KGS,lalafo
13001,Швея Прямострочка. 3-5 лет опыта. Таатан ТЦ,1.000000e+06,KGS,lalafo
4747,Низкая комиссия таксопарк онлайн подключение к такси работа в такси,6.666660e+05,KGS,lalafo
11823,Тигүүчү кыздар керек айлык акы жумасына берилет машынкалар баары жаңы,1.111111e+06,KGS,lalafo
12695,"Готов вложиться в хороший бизнес план до 100 000 usd, Отправляйте",1.000000e+05,USD,lalafo
11182,Требуется мастера сто,1.000000e+06,KGS,lalafo
8862,Мини цехке заказчик керек качественный количества беребиз дарегибиз,1.234568e+08,KGS,lalafo
8647,"Требуется Каменщик, Оплата Еженедельно, Менее года опыта",1.200000e+06,KGS,lalafo


In [9]:
df["salary_outlier"] = out
note("Зарплата < 10 тыс или > 600 тыс обнулена", "опечатки и сдельная оплата; вакансия сохранена",
     out.sum())
df.loc[out, "salary_kgs"] = np.nan
df["has_salary"] = df["salary_kgs"].notna()
df["salary_hidden"] = df["salary_type"] == "hidden"     # работодатель не указал зарплату
df["salary_kgs"].describe().round(0)

count      5840.0
mean      47796.0
std       36645.0
min       10000.0
25%       26000.0
50%       39000.0
75%       55000.0
max      535566.0
Name: salary_kgs, dtype: float64

## 6б. Исправление ошибки в справочнике сфер

В исходных данных каждая профессия (`sub_category`) относится ровно к одной сфере (`category`).
Профессия «Менеджер по продажам, менеджер по работе с клиентами» при этом отнесена к сфере
**«Спортивные клубы, фитнес, салоны красоты»**, хотя в ней сотни вакансий из продаж, банков,
дистрибуции. Это ошибка справочника, а не данных: профессия переносится в сферу
«Продажи, обслуживание клиентов». Остальные профессии оставлены как в источнике.

In [10]:
SALES = "Менеджер по продажам, менеджер по работе с клиентами"
wrong = (df["sub_category"] == SALES) & (df["category"] != "Продажи, обслуживание клиентов")
print(df.loc[wrong, "category"].value_counts())
note("Профессия «менеджер по продажам» перенесена в сферу «Продажи»", "ошибка справочника источника", wrong.sum())
df.loc[wrong, "category"] = "Продажи, обслуживание клиентов"

category
Спортивные клубы, фитнес, салоны красоты    563
Name: count, dtype: int64


## 7. Признаки

Готовим признаки один раз здесь, чтобы и анализ, и ML-репозиторий работали с одной таблицей.

In [11]:
# опыт: в данных уже корзины hh (0, 1–3, 3–6, 6+ лет)
df["experience"] = df["experience_start"].map(
    {0: "нет опыта", 1: "1–3 года", 3: "3–6 лет", 6: "6+ лет"})

# город: Бишкек сильно доминирует, остальное группируем
df["city"] = np.where(df["area"] == "Бишкек", "Бишкек",
             np.where(df["area"] == "Ош", "Ош", "Регионы"))

# навыки (почти только hh)
df["skills_list"] = df["skills"].fillna("").str.split(",").apply(
    lambda xs: [x.strip() for x in xs if x.strip()])
df["n_skills"] = df["skills_list"].str.len()

# образование из текста требований
df["education"] = np.select(
    [df["text"].str.contains(r"высше\w* образован|высшее\b|бакалавр|магистр|жогорку билим"),
     df["text"].str.contains(r"средне\w*[- ]?специальн|среднее профессиональн|колледж")],
    ["высшее", "среднее специальное"], default="не указано")

# требование к полу
female = df["text"].str.contains(r"девушк|женщин|аял киши|кыздар|\bкыз\b")
male = df["text"].str.contains(r"\bпарн|мужчин|эркек|жигит")
df["gender_req"] = np.select([female & ~male, male & ~female, female & male],
                             ["женщины", "мужчины", "оба указаны"], default="не указано")

# формат и условия
df["remote"] = df["text"].str.contains(r"удал[её]нн|remote|онлайн работ")
df["commission"] = df["text"].str.contains(r"процент|% от|бонус")
df["kyrgyz_text"] = df["text"].str.contains("[өүң]")
df["schedule"] = df["text"].str.extract(r"\b([1-7]/[1-7])\b")[0].fillna("не указан")
df["text_len"] = df["requirements"].fillna("").str.len()
df["pub_month"] = df["published"].dt.to_period("M").astype(str)

df[["experience", "city", "education", "gender_req", "remote", "commission", "schedule"]].describe()

,experience,city,education,gender_req,remote,commission,schedule
count,12646,12646,12646,12646,12646,12646,12646
unique,4,3,3,4,2,2,30
top,нет опыта,Бишкек,не указано,не указано,False,False,не указан
freq,5277,10547,11448,11466,12525,11879,11366


In [12]:
for c in ["education", "gender_req", "schedule"]:
    print(df[c].value_counts().head(6), "\n")

education
не указано             11448
высшее                  1146
среднее специальное       52
Name: count, dtype: int64 

gender_req
не указано     11466
женщины          877
мужчины          175
оба указаны      128
Name: count, dtype: int64 

schedule
не указан    11366
6/1            589
5/2            361
2/2            151
1/1             39
1/2             37
Name: count, dtype: int64 



## 8. Журнал очистки и сохранение

In [13]:
log_df = pd.DataFrame(log)
log_df.loc[len(log_df)] = {"шаг": "Итого вакансий", "почему": f"было {n0}", "строк": len(df)}
log_df

,шаг,почему,строк
0,Удалены резюме и объявления об услугах,"не вакансии, а предложения работников или услуги",338
1,Удалены дубликаты,одинаковые название + требования + зарплата; оставлена последняя публикация,365
2,Валюта не указана → KGS,объявления lalafo из КР,5
3,Пересчёт USD/RUR/EUR/KZT в сомы,курс НБКР на 14.03.2024,378
4,Ставка «hour» пересчитана в месяц,× 208,4
5,Ставка «day» пересчитана в месяц,× 26,1029
6,Ставка «week» пересчитана в месяц,× 4.33,65
7,Зарплата < 10 тыс или > 600 тыс обнулена,опечатки и сдельная оплата; вакансия сохранена,290
8,Профессия «менеджер по продажам» перенесена в сферу «Продажи»,ошибка справочника источника,563
9,Итого вакансий,было 13349,12646


In [14]:
print(f"Вакансий: {len(df)} | с правдоподобной зарплатой: {df['has_salary'].sum()} "
      f"({df['has_salary'].mean():.0%}) | скрыли зарплату: {df['salary_hidden'].mean():.0%}")

keep = ["id", "name", "company", "category", "sub_category", "source", "link", "published", "pub_month",
        "area", "city", "experience", "experience_start", "education", "gender_req", "schedule",
        "remote", "commission", "kyrgyz_text", "skills", "n_skills", "text_len",
        "requirements", "responsibilities",
        "compensation_from", "compensation_to", "compensation_currency",
        "salary_type", "pay_period", "salary_outlier", "salary_hidden", "has_salary", "salary_kgs"]
df[keep].to_csv("../data/processed/vacancies_clean.csv", index=False)
log_df.to_csv("../reports/cleaning_log.csv", index=False)
print("saved")

Вакансий: 12646 | с правдоподобной зарплатой: 5840 (46%) | скрыли зарплату: 52%


saved


## Выводы по очистке

> Перепишите своими словами и будьте готовы обосновать каждое правило на защите.

- Удалены резюме/услуги и дубликаты (см. журнал).
- Зарплата сведена к одному числу `salary_kgs` (сом/месяц): середина диапазона,
  курс НБКР, пересчёт дневной/недельной/почасовой оплаты.
- Выбросы обнулены, вакансии сохранены.
- Новые признаки: опыт, город, образование, пол, удалёнка, проценты, график, число навыков.
- Ограничения: правила по тексту ошибаются на части объявлений; 26 рабочих дней — допущение;
  курс зафиксирован на одну дату.